# Hyperparameter Tuning and Model Selection

> Use cross-validation to compare a small search space while keeping the test set reserved.

**Track:** Machine Learning Roadmap · **Language:** English

## Learning goals
- Distinguish parameters learned from data from hyperparameters chosen by the developer.
- Use `GridSearchCV` and `RandomizedSearchCV` with a pipeline.
- Read the selected parameters and the cross-validation score.
- Avoid tuning endlessly to a held-out test set.

In [1]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score

X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)
pipe = Pipeline([("scale", StandardScaler()), ("svc", SVC(probability=True, random_state=42))])
param_grid = {"svc__C": [.1, 1, 10], "svc__gamma": ["scale", .01, .1]}
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
search = GridSearchCV(pipe, param_grid, scoring="roc_auc", cv=cv, n_jobs=1, return_train_score=True)
search.fit(X_train, y_train)
print("Best params:", search.best_params_)
print("Best CV ROC-AUC:", round(search.best_score_,3))
print("Held-out test ROC-AUC:", round(roc_auc_score(y_test, search.best_estimator_.predict_proba(X_test)[:,1]),3))

Best params: {'svc__C': 10, 'svc__gamma': 0.01}
Best CV ROC-AUC: 0.996
Held-out test ROC-AUC: 0.998


In [2]:
cv_results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
display(cv_results[["param_svc__C", "param_svc__gamma", "mean_train_score", "mean_test_score", "std_test_score", "rank_test_score"]].head(8).round(3))

,param_svc__C,param_svc__gamma,mean_train_score,mean_test_score,std_test_score,rank_test_score
7,10.0,0.01,0.998,0.996,0.005,1
3,1.0,scale,0.998,0.995,0.004,2
4,1.0,0.01,0.996,0.994,0.005,3
6,10.0,scale,1.000,0.994,0.007,4
5,1.0,0.1,1.000,0.991,0.005,5
0,0.1,scale,0.991,0.989,0.005,6
1,0.1,0.01,0.990,0.988,0.006,7
8,10.0,0.1,1.000,0.986,0.009,8


## Search strategy and overfitting the validation process
- Grid search exhaustively checks a small grid; random search is often more efficient for many dimensions.
- Nested cross-validation can estimate the full tuning procedure more honestly when data are limited.
- Set random seeds where supported and record data versions, code version, search space, and scoring rules.
- Search results can be noisy; a tiny difference in CV score may not justify extra complexity.

## Practice
Add `svc__class_weight` as an option and compare ROC-AUC with average precision. Explain why a model selected using ROC-AUC may not be best for a rare-event workflow.